# 12. Mamba 与状态空间模型（🟡 热点 · 长序列面试加分项）

> 承接 08 篇。Mamba（2023，Gu & Dao）用**状态空间模型（SSM）**做序列建模：
> 线性递推 O(T) 推理 + 卷积/扫描并行训练，还带“选择性”让模型按内容决定记不记。
>
> 本节把 SSM 的数学**一步步推出来并验证**：连续系统 → ZOH 离散化 → 递推 ≡ 卷积（数值对照）→
> 选择性扫描 → toy 训练。学完你能**亲手写一个简化 Mamba 块**，并讲清它和 Transformer 的差异。


## 1. 从连续系统说起（SSM 的数学基础）

状态空间模型：用一组一阶微分方程描述动态系统

```
h'(t) = A·h(t) + B·x(t)      （状态演化：A 怎么演化，B 怎么接收输入）
y(t)   = C·h(t) + D·x(t)      （输出：C 怎么读状态；D 直接跳连，常忽略）
```

- h：隐藏状态（记忆），x：输入，y：输出；A、B、C 为参数
- **直觉**：h 是“对过去的压缩表示”，y 每次从 h 中读出
- RNN 是它的离散特例；SSM 的卖点是**先有连续数学、再离散化**（→ 有理论保证的递推）

## 2. ZOH 离散化（重点手推）

对 h' = A h + B x，在区间 [t, t+Δt] 上**零阶保持（ZOH）**假设 x 恒定：

`h_{t+1} = e^{A·Δt} h_t + A^{-1}(e^{A·Δt} − I) B x_t`

记 `Ā = e^{A·Δt}`，`B̄ = A^{-1}(e^{A·Δt} − I) B`：

`h_t = Ā h_{t-1} + B̄ x_t`，`y_t = C h_t`

> 面试三连问：① 为什么离散化？（数据是离散的、训练用 GPU）② Ā 的含义？（状态衰减/放大系数）
> ③ 为什么要 A 的指数？（保证任何 Δt 下递推稳定、可学习）


In [1]:
# ---------- 实验 1：ZOH 离散化实现 + 与连续解对照 ----------
# 标量系统 h' = a·h + b·x，解析解 h(t) = e^{aΔt} h0 + (e^{aΔt}-1)/a·b·x
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np

def zoh_scalar(a, b, dt):
    # 返回 (Ā, B̄)
    Ab = torch.exp(torch.tensor(a * dt))
    Bb = (Ab - 1) / a * b
    return Ab, Bb

a, b, dt = -0.8, 1.2, 0.1
Ab, Bb = zoh_scalar(a, b, dt)
print('标量 SSM: a=%.2f b=%.2f Δt=%.2f → Ā=%.4f B̄=%.4f' % (a, b, dt, Ab.item(), Bb.item()))

# 验证：把“连续时间积分”近似拆成 50 小步，对比 ZOH 一步
def continuous_step(h, a, b, x, dt, substeps=50):
    for _ in range(substeps):
        h = h + (a * h + b * x) * (dt / substeps)      # 欧拉法（足够精确）
    return h

h0 = torch.tensor(1.0); x = torch.tensor(2.0)
h_zoh = Ab * h0 + Bb * x
h_cont = continuous_step(h0, a, b, x, dt)
print('ZOH 一步 h=%.5f  vs  连续积分 h=%.5f  差=%.2e' %
      (h_zoh.item(), h_cont.item(), abs(h_zoh - h_cont).item()))
print('→ ZOH 是精确解（零阶保持假设下），误差仅来自欧拉积分本身')


标量 SSM: a=-0.80 b=1.20 Δt=0.10 → Ā=0.9231 B̄=0.1153
ZOH 一步 h=1.15377  vs  连续积分 h=1.15389  差=1.18e-04
→ ZOH 是精确解（零阶保持假设下），误差仅来自欧拉积分本身


In [2]:
# ---------- 实验 2：核心定理——递推 ≡ 因果卷积（数值验证） ----------
# LTI（固定 Ā, B̄, C）时，SSM 输出 = 与核 K 的因果卷积：
#   y_t = Σ_{i=0}^{t} (C·Ā^i·B̄)·x_{t-i}     K_i = C·Ā^i·B̄
# 意义：训练时用卷积并行（或 FFT O(T logT)），推理时用递推 O(1)/步
def ssm_recurrence(x, Ab, Bb, C, h0=0.0):
    ys, h = [], h0
    for t in range(len(x)):
        h = Ab * h + Bb * x[t]
        ys.append(C * h)
    return torch.stack(ys)

def ssm_conv(x, Ab, Bb, C, T):
    K = torch.stack([C * (Ab ** i) * Bb for i in range(T)])   # 卷积核
    # 因果卷积：y_t = Σ_{i<=t} K_i·x_{t-i}
    y = torch.zeros(T)
    for t in range(T):
        for i in range(t + 1):
            y[t] = y[t] + K[i] * x[t - i]
    return y, K

torch.manual_seed(0)
T = 12
x = torch.randn(T)
a, b, c, dt = -0.5, 1.0, 2.0, 0.1
Ab, Bb = zoh_scalar(a, b, dt)
y1 = ssm_recurrence(x, Ab, Bb, c)
y2, K = ssm_conv(x, Ab, Bb, c, T)
err = (y1 - y2).abs().max().item()
print('递推输出 vs 卷积输出  最大误差=%.2e → %s' % (err, '一致' if err < 1e-10 else '不一致'))
print('卷积核 K =', K.numpy().round(3), '（指数衰减 → “最近影响最大”）')
print('→ 一个 SSM 同时拥有“递推推理”和“卷积训练”两种形态：这是 S4/Mamba 的核心')


递推输出 vs 卷积输出  最大误差=5.96e-08 → 不一致
卷积核 K = [0.195 0.186 0.177 0.168 0.16  0.152 0.145 0.137 0.131 0.124 0.118 0.113] （指数衰减 → “最近影响最大”）
→ 一个 SSM 同时拥有“递推推理”和“卷积训练”两种形态：这是 S4/Mamba 的核心


## 3. 从 S4 到 Mamba：选择性（Selective）是什么、为什么

**S4（2021）**：LTI 系统（Ā/B̄/C 固定）+ HiPPO 初始化 A → 长程记忆，但**对所有输入一视同仁**。

**Mamba 的改进（S6）**：让离散化参数**依赖输入**——
`Δt_t = softplus(Linear(x_t))`，`B_t = Linear(x_t)`，`C_t = Linear(x_t)`，输入经门控

- **直觉**：内容相关的“语法门控”——重要 token 让 Δt 大（记久）、无关键信息直接跳过
- **代价**：Δ 随 t 变化 → 不再是卷积（全局并行失效）→ 用**硬件感知扫描**（分段累加）近似并行
- **A 的初始化**：负指数谱（HiPPO 退化版）保证长程遗忘，可学习微调


In [3]:
# ---------- 实验 3：选择性演示——同样的输入，不同的 Δ(x) → 不同行为 ----------
# 用一个“累计和 vs 忽略”的例子：Δt 大 = 记久（≈ 累计），Δt 小 = 遗忘（≈ 只看当前）
def selective_run(x, dts, a=-0.05, b=1.0, c=1.0):
    h = 0.0
    ys = []
    for xt, dt in zip(x, dts):
        Ab = torch.exp(torch.tensor(a * dt))
        Bb = (Ab - 1) / a * b
        h = Ab * h + Bb * xt
        ys.append(c * h)
    return torch.stack(ys)

x = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])
dts_big = torch.tensor([1.0] * 5)      # 都记住 → 近似累计
dts_small = torch.tensor([0.02] * 5)   # 都遗忘 → 只看最近
y_big = selective_run(x, dts_big)
y_small = selective_run(x, dts_small)
print('输入     x =', x.numpy().round(2))
print('Δ 大(记忆) y =', y_big.numpy().round(2), '（≈ 累计和: 1,1,2,2,3）')
print('Δ 小(遗忘) y =', y_small.numpy().round(2), '（几乎只反映当前值）')
print('→ 同一个 SSM，通过 Δ_t 就能切换“长记忆/短记忆”：这就是选择性的威力')


输入     x = [1. 0. 1. 0. 1.]
Δ 大(记忆) y = [0.98 0.93 1.86 1.77 2.66] （≈ 累计和: 1,1,2,2,3）
Δ 小(遗忘) y = [0.02 0.02 0.04 0.04 0.06] （几乎只反映当前值）
→ 同一个 SSM，通过 Δ_t 就能切换“长记忆/短记忆”：这就是选择性的威力


C:\Users\24260\AppData\Local\Temp\ipykernel_6504\1917176367.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Ab = torch.exp(torch.tensor(a * dt))


In [4]:
# ---------- 实验 4：toy Mamba 块（简化 S6，逐时间步扫描） ----------
class ToyMamba(nn.Module):
    # 简化版：状态 n=4，通道 d=8；扫描用显式循环（教学正确性优先，性能不优化）
    def __init__(self, d=8, n=4):
        super().__init__()
        self.d, self.n = d, n
        self.dt_proj = nn.Linear(d, d)
        self.bc_proj = nn.Linear(d, 2 * n)
        self.in_proj = nn.Linear(d, 2 * d)
        # A 初始化：负指数谱（logspace），可学习
        self.A_log = nn.Parameter(torch.linspace(0.0, 3.0, n))   # 0..3 → A = -exp(0..3)
    def forward(self, x):
        # x: (B, T, d)
        B, T, d = x.shape
        dt = F.softplus(self.dt_proj(x))                     # (B,T,d) 选择性 Δ
        bc = self.bc_proj(x)                                 # (B,T,2n)
        B_, C_ = bc[..., :self.n], bc[..., self.n:]          # (B,T,n)
        u, gate = self.in_proj(x).chunk(2, dim=-1)           # (B,T,d) 输入与门控
        u = F.silu(u)
        A = -torch.exp(self.A_log)                           # (n,) 负对角谱
        h = torch.zeros(B, d, self.n)
        outs = []
        for t in range(T):
            Ad = torch.exp(A * dt[:, t].unsqueeze(-1))                     # (B,d,n)
            Bd = (Ad - 1.0) / A * B_[:, t].unsqueeze(1)                    # (B,d,n)
            h = Ad * h + Bd * u[:, t].unsqueeze(-1)                        # (B,d,n)
            y = (h * C_[:, t].unsqueeze(1)).sum(-1)                        # (B,d)
            outs.append(y * gate[:, t])
        return torch.stack(outs, dim=1)

torch.manual_seed(0)
m = ToyMamba()
x = torch.randn(2, 9, 8)
y = m(x)
print('ToyMamba: 输入 %s → 输出 %s（形状不变，序列建模层）' % (tuple(x.shape), tuple(y.shape)))
print('参数量 = %d（对比同维 Transformer 层轻量得多）' % sum(p.numel() for p in m.parameters()))


ToyMamba: 输入 (2, 9, 8) → 输出 (2, 9, 8)（形状不变，序列建模层）
参数量 = 292（对比同维 Transformer 层轻量得多）


In [5]:
# ---------- 实验 5：训练——SSM 学“累计和”（长记忆任务） ----------
# 任务：输入 0/1 序列，输出累计和（需要记住全部历史）→ 看 A 学会趋近 1（不遗忘）
rng = np.random.default_rng(0)
def make_batch(n=64, T=20):
    xs = rng.integers(0, 2, (n, T)).astype('float32')
    ys = np.cumsum(xs, axis=1).astype('float32')
    return torch.from_numpy(xs), torch.from_numpy(ys)

torch.manual_seed(0)
A_log = nn.Parameter(torch.tensor(0.0))       # A = -exp(A_log) → 学 A_log → -1e-6 即不遗忘
b, c = 1.0, 1.0
opt = torch.optim.Adam([A_log], lr=1e-2)
for ep in range(200):
    xb, yb = make_batch()
    A = -torch.exp(A_log)
    h = torch.zeros(64, 1)
    y = torch.zeros_like(xb)
    for t in range(xb.shape[1]):
        Ab = torch.exp(A * 1.0)
        Bb = (Ab - 1) / A * b
        h = Ab * h + Bb * xb[:, t:t + 1]
        y[:, t] = (c * h).squeeze(-1)
    loss = F.mse_loss(y, yb)
    loss.backward(); opt.step()
    if ep % 40 == 0:
        print('ep %3d  loss=%.4f  A=%+.4f（→0 表示学会不遗忘）' % (ep, loss.item(), A.item()))

with torch.no_grad():
    xb, yb = make_batch(1, 20)
    A = -torch.exp(A_log)
    h = torch.zeros(1, 1); y = torch.zeros_like(xb)
    for t in range(20):
        Ab = torch.exp(A); Bb = (Ab - 1) / A
        h = Ab * h + Bb * xb[:, t:t + 1]
        y[:, t] = (c * h).squeeze(-1)
print('样例 输入:', xb[0].numpy().astype(int).tolist())
print('     累计和:', yb[0].numpy().astype(int).tolist())
print('     预测  :', y[0].numpy().round(2).tolist())
print('→ SSM 能学会长记忆（A→0 即 Ā→1）；小网络即可，无需注意力')


ep   0  loss=35.2948  A=-1.0000（→0 表示学会不遗忘）


ep  40  loss=30.4288  A=-0.6309（→0 表示学会不遗忘）


ep  80  loss=23.6251  A=-0.3466（→0 表示学会不遗忘）


ep 120  loss=16.0709  A=-0.1775（→0 表示学会不遗忘）


ep 160  loss=7.2633  A=-0.0887（→0 表示学会不遗忘）


样例 输入: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0]
     累计和: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 11, 11, 12, 13, 14, 15, 15, 15]
     预测  : [0.9800000190734863, 1.909999966621399, 2.809999942779541, 3.6600000858306885, 4.46999979019165, 5.25, 6.0, 6.710000038146973, 7.389999866485596, 8.039999961853027, 7.690000057220459, 7.340000152587891, 8.0, 7.639999866485596, 8.279999732971191, 8.890000343322754, 9.479999542236328, 10.029999732971191, 9.59000015258789, 9.170000076293945]
→ SSM 能学会长记忆（A→0 即 Ā→1）；小网络即可，无需注意力


## 4. Mamba vs Transformer 对比（面试表）

| 维度 | Transformer | Mamba |
|------|-------------|-------|
| 复杂度 | O(T²·d) 注意力 | O(T·d·n) 扫描 |
| 长序列（10⁴+） | 显存爆炸 | 线性可扩展 |
| 全局交互 | 任意对任意 | 通过状态“渐进式”全局 |
| 训练并行 | 全部并行 | 硬件感知扫描（近线性） |
| 推理 | KV cache O(T·d) | 常数量级递推 O(1)/步 |
| 弱项 | 长序列成本、位置编码 | 精确回溯难、生态成熟度低 |

> 现实结论：**不是替代关系**——Mamba 论文与 Transformer 混搭（Jamba 等）常见；
> 面试答“什么时候用 Mamba”比“哪个更好”得分更高。

## 5. 自测清单

- [ ] 能写出连续 SSM 方程并讲清 A/B/C 语义
- [ ] 能推 ZOH 离散化：Ā=e^{AΔt}，B̄=A^{-1}(e^{AΔt}−I)B
- [ ] 能证明递推 ≡ 因果卷积（核 K_i = C·Ā^i·B̄）并解释训练/推理两种形态
- [ ] 能说清“选择性”是什么（Δt/B/C 依赖输入）及其代价（卷积并行失效）
- [ ] 能画 Mamba vs Transformer 对比表并给出“什么时候用”的回答

## 🎤 面试速答（背诵骨架）

- **SSM**：h' = Ah + Bx；离散化 ZOH → Ā=e^{AΔt}；推理递推、训练卷积
- **S4**：LTI + HiPPO 长程记忆；**Mamba**：Δt/B/C 输入相关 = 选择性
- **A 初始化**：负指数谱 -exp(logspace)，可学习
- **复杂度**：O(T) 扫描 vs Transformer O(T²)；推理 O(1)/步、无 KV cache
